## Import Libraries

In [ ]:
import nltk 
import re 
import ast
import html
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
%pip install python-docx
from docx import Document
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer 
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer 

## Load all Datasets

In [ ]:
animes = pd.read_csv("animes.csv")
profiles = pd.read_csv("profiles.csv")
reviews = pd.read_csv("reviews.csv")

## Data Description and Relationships

In [ ]:
datasets = {
    'Animes': animes,
    'Profiles': profiles,
    'Reviews': reviews
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name + ' Dataset':^40}")
    print(f"{'='*40}")

    print("\nFirst 5 Rows:")
    display(df.head())
    
    print("\nDataset Information:")
    df.info()

    print("\nDescriptive Statistics:")
    display(df.describe(include='all'))

    #print("\nShape:", df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

## Data Cleaning and Preparation

In [ ]:
# Duplicate handling

cleaning_summary = []

def clean_duplicates(df, name, key):
    original = len(df)
    exact = int(df.duplicated().sum())
    no_exact = df.drop_duplicates().copy()
    repeated_ids = int(no_exact.duplicated(subset=[key]).sum())
    cleaned = no_exact.drop_duplicates(subset=[key], keep="first").copy()
    cleaning_summary.append({
        "Dataset": name, "Original Rows": original,
        "Exact Duplicates Removed": exact,
        "Additional Duplicate IDs Removed": repeated_ids,
        "Cleaned Rows": len(cleaned)
    })
    return cleaned

animes_clean = clean_duplicates(animes, "Anime", "uid")
profiles_clean = clean_duplicates(profiles, "Profiles", "profile")
reviews_clean = clean_duplicates(reviews, "Reviews", "uid")

cleaning_summary_df = pd.DataFrame(cleaning_summary)
display(cleaning_summary_df)
print("Cleaned shapes:", animes_clean.shape, profiles_clean.shape, reviews_clean.shape)

In [ ]:
# Missing Values and Numeric Consistency

for col in ["uid", "episodes", "members", "popularity", "ranked", "score"]:
    animes_clean[col] = pd.to_numeric(animes_clean[col], errors="coerce")
for col in ["uid", "anime_uid", "score"]:
    reviews_clean[col] = pd.to_numeric(reviews_clean[col], errors="coerce")

print("Anime missing values after conversion:")
display(animes_clean.isna().sum().to_frame("Missing"))
print("Reviews missing values after conversion:")
display(reviews_clean.isna().sum().to_frame("Missing"))

In [ ]:
# Load stopwords
with open("English stopwords", "r", encoding="utf-8") as file:
    stop_words = set(line.strip() for line in file)

stop_words.add("etc")

# Preserve negation words
stop_words -= {"no", "not", "nor", "never"}


def preprocess(text):

    if pd.isna(text):
        return ""

    # Convert HTML entities
    text = html.unescape(str(text))

    # Remove anime review rating headers
    text = re.sub(
        r'^\s*(?:more\s+pics\s*)?Overall\s*\d+\s*'
        r'Story\s*\d+\s*Animation\s*\d+\s*'
        r'Sound\s*\d+\s*Character\s*\d+\s*'
        r'Enjoyment\s*\d+',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove stopwords
    tokens = [
        word for word in text.split()
        if word not in stop_words
    ]

    return " ".join(tokens)


# Apply preprocessing
reviews_clean["processed_text"] = (
    reviews_clean["text"].apply(preprocess)
)

# Display results
display(
    reviews_clean[["text", "processed_text"]].head(3)
)

## Exploratory Data Analysis (EDA)

In [ ]:
# Histograms: anime scores, episodes, popularity and review scores

# Set a clean visual style
sns.set_theme(style="whitegrid", palette="muted")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Anime Score Distribution
sns.histplot(
    animes_clean["score"].dropna(),
    bins=30,
    color="#547AA5",
    edgecolor="white",
    ax=axes[0, 0]
)
axes[0, 0].set_title("Distribution of Anime Scores", fontsize=14, weight="bold")
axes[0, 0].set_xlabel("Anime Score")
axes[0, 0].set_ylabel("Number of Anime")

# 2. Episode Count Distribution
# Show up to the 99th percentile for readability
episode_limit = animes_clean["episodes"].quantile(0.99)

sns.histplot(
    animes_clean.loc[
        animes_clean["episodes"] <= episode_limit, "episodes"
    ].dropna(),
    bins=35,
    color="#69A69A",
    edgecolor="white",
    ax=axes[0, 1]
)
axes[0, 1].set_title("Episode Counts (Up to 99th Percentile)",
                     fontsize=14, weight="bold")
axes[0, 1].set_xlabel("Number of Episodes")
axes[0, 1].set_ylabel("Number of Anime")

# 3. Popularity Ranking Distribution
sns.histplot(
    animes_clean["popularity"].dropna(),
    bins=30,
    color="#B38BC2",
    edgecolor="white",
    ax=axes[1, 0]
)
axes[1, 0].set_title("Distribution of Popularity Rankings",
                     fontsize=14, weight="bold")
axes[1, 0].set_xlabel("Popularity Rank (Lower = More Popular)")
axes[1, 0].set_ylabel("Number of Anime")

# 4. Review Score Distribution
sns.histplot(
    reviews_clean["score"].dropna(),
    bins=range(0, 13),
    color="#E7A36C",
    edgecolor="white",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Distribution of Review Scores",
                     fontsize=14, weight="bold")
axes[1, 1].set_xlabel("Review Score")
axes[1, 1].set_ylabel("Number of Reviews")
axes[1, 1].set_xticks(range(0, 12))

# Improve spacing and appearance
for ax in axes.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2)
    ax.grid(axis="x", visible=False)

plt.suptitle(
    "Anime Dataset: Exploratory Data Analysis",
    fontsize=18,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

## Exploratory Data Analysis

In [ ]:
# Top 10 genres. Handles comma-separated strings and list-like strings.
def parse_genres(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    text = str(value).strip()
    if text.startswith("["):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple)):
                return [str(g).strip() for g in parsed if str(g).strip()]
        except (ValueError, SyntaxError):
            pass
    return [g.strip(" []'\"") for g in text.split(",") if g.strip(" []'\"")]

genres_long = animes_clean[["uid", "score", "genre"]].copy()
genres_long["genre"] = genres_long["genre"].apply(parse_genres)
genres_long = genres_long.explode("genre").dropna(subset=["genre"])
top_genres = genres_long["genre"].value_counts().head(10)
ax = top_genres.sort_values().plot.barh(figsize=(9, 5), title="Top 10 Anime Genres")
ax.set_xlabel("Number of Anime")
plt.tight_layout(); plt.show()

In [ ]:
# Correlation matrix and score relationships
numeric_cols = ["score", "popularity", "members", "episodes", "ranked"]
corr = animes_clean[numeric_cols].corr()
display(corr)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_cols)), numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.ravel(), ["popularity", "members", "episodes", "ranked"]):
    ax.scatter(animes_clean[col], animes_clean["score"], alpha=0.18, s=9)
    ax.set_xlabel(col)
    ax.set_ylabel("Anime Score")
    ax.set_title(f"Score vs {col.title()}")
plt.tight_layout(); plt.show()

## Text Preprocessing

## Feature Engineering

In [ ]:
# Function to convert genre strings into lists
def parse_genres(value):
    if pd.isna(value) or not str(value).strip():
        return []

    try:
        result = ast.literal_eval(str(value))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

## Exploratory Data Analysis (EDA)

In [ ]:
# Create the anime-level feature table
anime_features = animes_clean.copy()

# Review features: one row per anime ID
review_summary = (
    reviews_clean.groupby("anime_uid", dropna=True)
    .agg(review_count=("uid", "count"), avg_review_score=("score", "mean"))
    .reset_index()
)

# Popularity index: lower original rank is better
min_rank = anime_features["popularity"].min()
max_rank = anime_features["popularity"].max()
if pd.notna(min_rank) and pd.notna(max_rank) and max_rank > min_rank:
    anime_features["popularity_index"] = (
        (max_rank - anime_features["popularity"]) / (max_rank - min_rank) * 100
    )
else:
    anime_features["popularity_index"] = np.nan

# Genre count
anime_features["genre_count"] = anime_features["genre"].apply(
    lambda value: len(parse_genres(value))
)

# Heuristic franchise flag: title mentions sequels or later installments
franchise_pattern = r"\b(?:season\s*\d+|\d+(?:st|nd|rd|th)\s+season|part\s*\d+|sequel)\b"
anime_features["franchise_indicator"] = (
    anime_features["title"].fillna("")
    .str.contains(franchise_pattern, case=False, regex=True)
    .astype(int)
)

# Episode count is already present: numeric conversion was performed above.

# Merge review aggregates into anime; preserve all anime rows
final_dataset = anime_features.merge(
    review_summary, left_on="uid", right_on="anime_uid",
    how="left", validate="one_to_one"
).drop(columns=["anime_uid"])
final_dataset["review_count"] = final_dataset["review_count"].fillna(0).astype(int)
# avg_review_score remains NaN when there are no matching reviews.
print("Anime + review features:", final_dataset.shape)
display(final_dataset[["title", "score", "episodes", "popularity_index", "genre_count",
                       "franchise_indicator", "review_count", "avg_review_score"]].head())

In [ ]:
# Profile favorites feature

# favorites_anime contains strings such as "['33352', '25013']"
def parse_favorites(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []

profiles_fav = profiles_clean[["profile", "favorites_anime"]].copy()
profiles_fav["favorites_anime"] = profiles_fav["favorites_anime"].apply(parse_favorites)
favorites = profiles_fav.explode("favorites_anime")
favorites["favorites_anime"] = pd.to_numeric(favorites["favorites_anime"], errors="coerce")
favorites = favorites.dropna(subset=["favorites_anime", "profile"])
favorites = favorites.drop_duplicates(subset=["profile", "favorites_anime"])

favorites_summary = (
    favorites.groupby("favorites_anime")["profile"]
    .nunique().reset_index(name="favorites_count")
)

final_dataset = final_dataset.merge(
    favorites_summary, left_on="uid", right_on="favorites_anime",
    how="left", validate="one_to_one"
).drop(columns=["favorites_anime"])
final_dataset["favorites_count"] = final_dataset["favorites_count"].fillna(0).astype(int)

print("Final dataset shape:", final_dataset.shape)
print("Duplicate anime IDs:", final_dataset["uid"].duplicated().sum())
print("Total matched profile–anime favorites:", final_dataset["favorites_count"].sum())
display(final_dataset.head())

## Data Dictionary

In [ ]:
# 1. Create output folder
OUTPUT_DIR = Path("Anime_Project_Output2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Generate data dictionary from final dataset
definitions = {
    "uid": "Unique anime identifier",
    "title": "Anime title",
    "synopsis": "Anime description",
    "genre": "Anime genre labels",
    "aired": "Release or broadcast information",
    "episodes": "Number of episodes",
    "members": "Number of anime community members",
    "popularity": "Popularity ranking; lower is better",
    "ranked": "Overall anime ranking; lower is better",
    "score": "Overall anime rating",
    "img_url": "Anime image URL",
    "link": "Anime information URL",
    "popularity_index": "Normalized popularity score (0-100)",
    "genre_count": "Number of genres assigned",
    "franchise_indicator": "Possible related-release title flag",
    "review_count": "Number of reviews for each anime",
    "avg_review_score": "Average valid review score",
    "review_score_std": "Variation in review scores",
    "favorites_count": "Number of profiles favoriting the anime"
}

data_dictionary = pd.DataFrame({
    "Column": final_dataset.columns,
    "Data Type": [
        str(final_dataset[c].dtype)
        for c in final_dataset.columns
    ],
    "Description": [
        definitions.get(c, "Description needed")
        for c in final_dataset.columns
    ],
    "Missing Values": [
        final_dataset[c].isna().sum()
        for c in final_dataset.columns
    ]
})
# 3. Save final merged and clean datasets
final_dataset.to_csv(
    OUTPUT_DIR / "final_anime_dataset.csv",
    index=False
)
# Export the three cleaned datasets
animes_clean.to_csv(
    OUTPUT_DIR / "anime_clean.csv",
    index=False
)

profiles_clean.to_csv(
    OUTPUT_DIR / "profiles_clean.csv",
    index=False
)

reviews_clean.to_csv(
    OUTPUT_DIR / "reviews_clean.csv",
    index=False
)

# 4. Save data dictionary as Word document
doc = Document()
doc.add_heading("Anime Acquisition - Data Dictionary", 0)

table = doc.add_table(rows=1, cols=4)
table.style = "Table Grid"

for i, heading in enumerate(data_dictionary.columns):
    table.rows[0].cells[i].text = heading

for _, row in data_dictionary.iterrows():
    cells = table.add_row().cells
    for i, value in enumerate(row):
        cells[i].text = str(value)

doc.save(OUTPUT_DIR / "anime_data_dictionary.docx")

# 5. Calculate current dataset statistics
anime_rows = len(animes_clean)
review_rows = len(reviews_clean)
profile_rows = len(profiles_clean)
final_rows, final_cols = final_dataset.shape
duplicate_ids = final_dataset["uid"].duplicated().sum()

favorites_total = (
    int(final_dataset["favorites_count"].sum())
    if "favorites_count" in final_dataset.columns else 0
)

# 6. Generate cleaning report
report = f"""
ANIME ACQUISITION - DATA CLEANING REPORT

Cleaned datasets:
Anime: {anime_rows:,}
Reviews: {review_rows:,}
Profiles: {profile_rows:,}

Cleaning actions:
- Removed duplicate records and identifiers.
- Reviewed missing values and numerical data types.
- Checked scores and episode counts for invalid values.
- Parsed genre and favorite anime lists.
- Created popularity index and genre counts.
- Created exploratory franchise indicators.
- Calculated review counts and average review scores.
- Calculated distinct profile-anime favorite counts.
- Merged the three datasets into one anime-level dataset.
- Retained potential outliers for further investigation.

Validation:
- Matched favorite relationships: {favorites_total:,}
- Duplicate anime IDs: {duplicate_ids:,}

Final dataset:
Rows: {final_rows:,}
Columns: {final_cols:,}

Remaining missing values:
"""

for col, count in final_dataset.isna().sum().items():
    if count > 0:
        report += f"- {col}: {count:,}\n"

report += """
Limitations:
- Data may represent different collection snapshots.
- Favorites cover only the available user profiles.
- Some anime have no available review scores.
- Popularity index is relative to this dataset.
- Franchise indicators are unverified title-based estimates.
- Extreme episode counts and membership values were retained.
- Licensing costs and ownership rights are unavailable.
"""

# 7. Save cleaning report as Word document
report_doc = Document()
report_doc.add_heading(
    "Anime Acquisition - Data Cleaning Report", 0
)

for line in report.strip().split("\n"):
    if not line.strip():
        continue
    if line.endswith(":"):
        report_doc.add_heading(line, level=2)
    elif line.startswith("- "):
        report_doc.add_paragraph(line[2:], style="List Bullet")
    else:
        report_doc.add_paragraph(line)

report_doc.save(
    OUTPUT_DIR / "anime_cleaning_report.docx"
)

# 8. Export duplicate cleaning summary if available
if "cleaning_summary_df" in globals():
    cleaning_summary_df.to_csv(
        OUTPUT_DIR / "duplicate_cleaning_summary.csv",
        index=False
    )

# 9. Confirm all exported files
print("ALL PROJECT DELIVERABLES EXPORTED SUCCESSFULLY!")
print("\nSaved in:", OUTPUT_DIR.resolve())

for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)
